In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error ,r2_score

df = pd.read_csv('uber.csv')

# preprocessing
df.info

df.describe()

df.head(5)

print(df.isnull().sum())

df = df.dropna()

print(df.isnull().sum())

df.dtypes

# removing outliers
sns.boxplot(x=df['fare_amount'])
plt.title("Fare Amount -Before Outlier")
plt.show()

Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)
IQR = Q3-Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Lower limit:",lower , "Upper limit:",upper)
df = df[(df['fare_amount']>=lower) & (df['fare_amount'] <=upper)]
# removing outliers

sns.boxplot(x = df['fare_amount'])
plt.title('Fare amount - after outlier')
plt.show()

#distance
dlat = df['dropoff_latitude'] - df['pickup_latitude']
dlong = df['dropoff_longitude'] - df['pickup_longitude']
df["distance"] = (dlat**2 + dlong**2) ** 0.5

#filtering
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 100)]
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <=6)]
df = df[df['distance'] < 5]

# extract time features
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek

corr = df[['fare_amount','distance','passenger_count','day_of_week','hour']].corr()
sns.heatmap(corr , annot=True, cmap = 'coolwarm')
plt.title(" Correlation Matrix ")
plt.show()

X = df[['distance','passenger_count','hour','day_of_week']]
y = df['fare_amount']

X_train , X_test , y_train , y_test = train_test_split(
    X,y,test_size = 0.2 , random_state = 42)
lr = LinearRegression()
lr.fit(X_train,y_train)
y_pred_lr =  lr.predict(X_test)

rf= RandomForestRegressor()
rf.fit(X_train,y_train)
y_pred_rf = rf.predict(X_test)

def evaluate(y_true ,y_pred,model_name):
    rmse = np.sqrt(mean_squared_error(y_true,y_pred))
    r2 =r2_score(y_true,y_pred)
    print(f"{model_name} Results:")
    print(f"R2 score: {r2:.6f}")
    print(f"RMSE: {rmse:.6f}\n")

evaluate(y_test,y_pred_lr,"Linear Regression")
evaluate(y_test,y_pred_rf,"RandomForest Regression")
